In [43]:
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import InMemoryVectorStore
from langchain.tools import tool
from langchain.agents import create_agent

In [45]:
loader = PyPDFLoader("../data/medical_report.pdf")
docs = loader.load()
len(docs)

9

In [46]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
splitted_docs = splitter.split_documents(docs)
len(splitted_docs)

26

In [ ]:
embeddings = OllamaEmbeddings(model="nomic-embed-text")
vector_store = InMemoryVectorStore.from_documents(
    documents = splitted_docs,
    embedding = embeddings
)

In [ ]:
#### agent = tools, llm, prompt

In [36]:
@tool
def retriever_tool(query:str):
    """
        This tool can help you to retrieve the data of the PDF Documents, 
        and these documents have details about medical reports.
    """
    print("Tool Called", query)

    docs = vector_store.similarity_search(query = query, k=4)
    context = ""

    for doc in docs:
        context += doc.page_content + "\n\n"

    return context

In [30]:
llm = ChatOllama(model= "phi4-mini")

In [ ]:
System_Prompt = """
    You are helpful assistant that answers questions using retrieved context.
    Always use the 'retriever_tool' too for questions requiring extgerna knowledge.
"""

In [ ]:
agent = create_agent(
    model = llm,
    tools = [retriever_tool],
    system_prompt= System_Prompt
)

In [ ]:
query = "What is the name of patient, and what is name of Doctors"
response = agent.invoke({"messages":[{"role":"user","content":query}]})

In [ ]:
result = response["messages"][-1].content

In [ ]:
print(result)